# Interactive Algorithm Comparison Dashboard

Click legend entries to toggle algorithms. Double-click to isolate one.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

run = [
    'a_20260425_174404',
    # 'aisle_first'
]
"""
Supported values for `run`:
- str: single run name (e.g., 'a_20260421_195503') or direct CSV path
- list[str]: multiple run names and/or direct CSV paths
- []: auto-detect and load all CSV files under results/
- None: auto-detect the latest CSV file under results/
"""

results_candidates = [Path('results'), Path('../results')]
results_dir = next((p for p in results_candidates if p.exists() and p.is_dir()), None)
if results_dir is None:
    searched = [str(p.resolve()) for p in results_candidates]
    raise FileNotFoundError(f"Results folder not found. Searched: {searched}")

def resolve_csv_path(entry: str) -> Path:
    entry_path = Path(entry)
    # If a direct CSV path is provided, try it as-is and relative to the results folder.
    if entry_path.suffix.lower() == '.csv' or '/' in entry or '\\' in entry:
        direct_candidates = [entry_path, results_dir / entry_path]
        for candidate in direct_candidates:
            if candidate.exists() and candidate.is_file():
                return candidate

        searched = [str(p.resolve()) for p in direct_candidates]
        raise FileNotFoundError(f"CSV not found for run entry {entry!r}. Searched: {searched}")

    run_dir = results_dir / entry
    if run_dir.exists() and run_dir.is_dir():
        summary_candidates = sorted(run_dir.glob('summary_*.csv'))
        if summary_candidates:
            return summary_candidates[0]  # first summary_ file in the folder

    # Fallback to legacy expected filename.
    direct_candidates = [run_dir / f'summary_{entry}.csv']
    for candidate in direct_candidates:
        if candidate.exists() and candidate.is_file():
            return candidate

    searched = [str(p.resolve()) for p in direct_candidates]
    raise FileNotFoundError(f"CSV not found for run entry {entry!r}. Searched: {searched}")

csv_files: list[Path]

if run is None:
    csv_candidates = sorted(results_dir.rglob('*.csv'))
    if not csv_candidates:
        raise FileNotFoundError(f"No .csv files found inside {results_dir.resolve()}")
    csv_files = [csv_candidates[-1]]  # latest by sorted path
elif isinstance(run, str):
    csv_files = [resolve_csv_path(run)]
elif isinstance(run, list):
    if len(run) == 0:
        csv_files = sorted(results_dir.rglob('*.csv'))
        if not csv_files:
            raise FileNotFoundError(f"No .csv files found inside {results_dir.resolve()}")
    else:
        csv_files = [resolve_csv_path(entry) for entry in run]
else:
    raise TypeError('`run` must be None, str, or list[str].')

print('Using CSV files:')
for file_path in csv_files:
    print(f'- {file_path}')

df = pd.concat([pd.read_csv(path) for path in csv_files], ignore_index=True)

required_columns = ['instance', 'algorithm', 'objective_mean', 'exec_time_mean']
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

best_objectives_candidates = [
    Path('best_solutions') / 'best_objectives.csv',
    Path('../best_solutions') / 'best_objectives.csv',
]
best_objectives_path = next((p for p in best_objectives_candidates if p.exists()), None)
if best_objectives_path is None:
    searched = [str(p.resolve()) for p in best_objectives_candidates]
    raise FileNotFoundError(f"Best objectives file not found. Searched: {searched}")

best_df = pd.read_csv(best_objectives_path)
if 'instance' not in best_df.columns or 'best_objective' not in best_df.columns:
    raise ValueError("best_objectives.csv must contain columns: instance, best_objective")

if 'dataset' in df.columns and 'dataset' in best_df.columns:
    merge_columns = ['dataset', 'instance']
else:
    merge_columns = ['instance']

df = df.merge(best_df[merge_columns + ['best_objective']], on=merge_columns, how='left')
ordered_columns = [col for col in df.columns if col != 'best_objective'] + ['best_objective']
df = df[ordered_columns]


def add_legend_mass_toggle(fig: go.Figure, y_position: float = 1.18) -> None:
    trace_count = len(fig.data)
    fig.update_layout(
        updatemenus=[
            dict(
                type='buttons',
                direction='right',
                x=1.0,
                y=y_position,
                xanchor='right',
                yanchor='bottom',
                showactive=False,
                buttons=[
                    dict(
                        label='Selecionar todos',
                        method='update',
                        args=[{'visible': [True] * trace_count}]
                    ),
                    dict(
                        label='Remover todos',
                        method='update',
                        args=[{'visible': ['legendonly'] * trace_count}]
                    ),
                ],
            )
        ]
    )


df.head()

Using CSV files:
- ../results/a_20260425_174404/summary_a_20260425_174404.csv


,dataset,algorithm,instance,total_runs,feasible_runs,infeasible_runs,timed_out_runs,objective_mean,objective_median,objective_mode,...,aisles_variance,aisles_std_dev,exec_time_mean,exec_time_median,exec_time_mode,exec_time_min,exec_time_max,exec_time_variance,exec_time_std_dev,best_objective
0,a,af_useful_order-desc_prune-multi,instance_0001.txt,1,1,0,0,15.000000,15.000000,15.000000,...,0.0,0.0,0.000646,0.000646,0.000646,0.000646,0.000646,0.0,0.0,15.000
1,a,af_useful_order-desc_prune-multi,instance_0002.txt,1,1,0,0,2.000000,2.000000,2.000000,...,0.0,0.0,0.000146,0.000146,0.000146,0.000146,0.000146,0.0,0.0,2.000
2,a,af_useful_order-desc_prune-multi,instance_0003.txt,1,1,0,0,6.888889,6.888889,6.888889,...,0.0,0.0,0.003438,0.003438,0.003438,0.003438,0.003438,0.0,0.0,12.000
3,a,af_useful_order-desc_prune-multi,instance_0004.txt,1,1,0,0,3.500000,3.500000,3.500000,...,0.0,0.0,0.002954,0.002954,0.002954,0.002954,0.002954,0.0,0.0,3.500
4,a,af_useful_order-desc_prune-multi,instance_0005.txt,1,1,0,0,149.680000,149.680000,149.680000,...,0.0,0.0,0.147280,0.147280,0.147280,0.147280,0.147280,0.0,0.0,177.875


In [2]:
# Objective Mean as % of Best Objective

objective_plot = (
    df.pivot_table(
        index='instance',
        columns='algorithm',
        values='objective_mean',
        aggfunc='mean'
    )
    .sort_index()
    .fillna(0)
)

best_objective_by_instance = (
    df[['instance', 'best_objective']]
    .drop_duplicates(subset='instance')
    .set_index('instance')['best_objective']
)

objective_pct_plot = objective_plot.div(best_objective_by_instance, axis=0) * 100
objective_pct_plot = objective_pct_plot.replace([float('inf'), -float('inf')], 0).fillna(0)

fig = go.Figure()
for algo in objective_pct_plot.columns:
    fig.add_trace(go.Bar(
        name=algo,
        x=objective_pct_plot.index,
        y=objective_pct_plot[algo],
        hovertemplate=f'<b>{algo}</b><br>Instance: %{{x}}<br>%{{y:.1f}}% of best<extra></extra>'
    ))

fig.add_hline(y=100, line_dash='dash', line_color='black', annotation_text='best (100%)')

fig.update_layout(
    barmode='group',
    title='Objective Mean by Instance and Algorithm (% of Best Objective)',
    xaxis_title='Instance',
    yaxis_title='% of best_objective',
    legend_title='Algorithm',
    height=700,
    margin=dict(t=150),
    xaxis_tickangle=-45,
)
add_legend_mass_toggle(fig)
fig.show()

In [3]:
# Execution Time Mean

exec_time_plot = (
    df.pivot_table(
        index='instance',
        columns='algorithm',
        values='exec_time_mean',
        aggfunc='mean'
    )
    .sort_index()
    .fillna(0)
)

fig = go.Figure()
for algo in exec_time_plot.columns:
    fig.add_trace(go.Bar(
        name=algo,
        x=exec_time_plot.index,
        y=exec_time_plot[algo],
        hovertemplate=f'<b>{algo}</b><br>Instance: %{{x}}<br>%{{y:.4f}}s<extra></extra>'
    ))

fig.update_layout(
    barmode='group',
    title='Execution Time Mean by Instance and Algorithm',
    xaxis_title='Instance',
    yaxis_title='exec_time_mean (s)',
    legend_title='Algorithm',
    height=600,
    margin=dict(t=150),
    xaxis_tickangle=-45,
)
add_legend_mass_toggle(fig)
fig.show()

In [4]:
# Rankings

ranking_dataset_param = None  # e.g., 'a', 'b', 'x' or None for all datasets

ranking_base = df[['instance', 'algorithm', 'objective_mean', 'exec_time_mean']].copy()
if ranking_dataset_param is not None and 'dataset' in df.columns:
    ranking_base = df[df['dataset'] == ranking_dataset_param][
        ['instance', 'algorithm', 'objective_mean', 'exec_time_mean']
    ].copy()

if ranking_base.empty:
    raise ValueError(f'No rows found for dataset={ranking_dataset_param!r}')


def build_final_ranking(base_df: pd.DataFrame, metric_column: str, ascending: bool) -> pd.DataFrame:
    metric_df = base_df.dropna(subset=[metric_column]).copy()
    if metric_df.empty:
        raise ValueError(f'No non-null values found for metric {metric_column!r}.')

    metric_df['N'] = metric_df.groupby('instance')['algorithm'].transform('nunique')
    metric_df['rank_class'] = metric_df.groupby('instance')[metric_column].rank(
        method='min', ascending=ascending
    ).astype(int)
    metric_df['score'] = metric_df['N'] - metric_df['rank_class']

    final_ranking = (
        metric_df.groupby('algorithm', as_index=False)['score']
        .agg(total_score='sum', average_score='mean', instances_count='count')
        .sort_values(['total_score', 'average_score', 'algorithm'], ascending=[False, False, True])
        .reset_index(drop=True)
    )
    final_ranking['final_rank'] = final_ranking['total_score'].rank(
        method='min', ascending=False
    ).astype(int)

    return final_ranking[
        ['final_rank', 'algorithm', 'total_score', 'average_score', 'instances_count']
    ]


objective_ranking = build_final_ranking(
    ranking_base, metric_column='objective_mean', ascending=False
)
exec_time_ranking = build_final_ranking(
    ranking_base, metric_column='exec_time_mean', ascending=True
)

print(f'Dataset filter for ranking: {ranking_dataset_param!r}')
print('\nOBJECTIVE_MEAN - FINAL RANKING')
display(objective_ranking)
print('\nEXEC_TIME_MEAN - FINAL RANKING')
display(exec_time_ranking)

Dataset filter for ranking: None

OBJECTIVE_MEAN - FINAL RANKING


,final_rank,algorithm,total_score,average_score,instances_count
0,1,af_useful_order-desc_prune-multi,48,2.4,20
1,1,ga_base_uniform_seeded_useful_aisle,48,2.4,20
2,3,ga_full,34,1.7,20
3,4,ga_full_25,30,1.5,20



EXEC_TIME_MEAN - FINAL RANKING


,final_rank,algorithm,total_score,average_score,instances_count
0,1,ga_full_25,51,2.55,20
1,2,ga_full,39,1.95,20
2,3,af_useful_order-desc_prune-multi,34,1.70,20
3,4,ga_base_uniform_seeded_useful_aisle,9,0.45,20
